# 04 — EDA Supporting Evidence

Notebook này chứa các phân tích phụ để củng cố lập luận trong GitHub/appendix. Không cần đưa toàn bộ vào report 4 trang, nhưng nên giữ trên repo để chứng minh nhóm đã khám phá dữ liệu rộng và có kiểm chứng.

Nhóm chart phụ:

- traffic quality và conversion proxy;
- geo / region contribution;
- customer cohort / repeat behavior;
- payment & cancellation diagnostics;
- fulfillment lead time;
- review quality and returns;
- anomaly detection trên daily sales.

In [1]:
# ============================================================
# Shared setup
# ============================================================
from pathlib import Path
import os
import sys
import json
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 120)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

# The notebooks are designed to run from repo/notebooks.
# They also work in Colab if you point PROJECT_ROOT or DATA_ROOT manually.
try:
    NOTEBOOK_DIR = Path.cwd().resolve()
except Exception:
    NOTEBOOK_DIR = Path(".").resolve()

PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR

RAW_DIR = PROJECT_ROOT / "data" / "raw"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
MARTS_DIR = PROJECT_ROOT / "data" / "marts"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
REPORTS_DIR = PROJECT_ROOT / "reports"
FIGURES_DIR = REPORTS_DIR / "figures"
TABLES_DIR = REPORTS_DIR / "tables"

for d in [RAW_DIR, INTERIM_DIR, MARTS_DIR, PROCESSED_DIR, FIGURES_DIR, TABLES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

SEARCH_DIRS = [RAW_DIR, INTERIM_DIR, MARTS_DIR, PROCESSED_DIR, PROJECT_ROOT, Path("/mnt/data")]

def find_csv(filename: str) -> Path:
    """Find a CSV across standard repo folders and /mnt/data fallback."""
    for d in SEARCH_DIRS:
        p = d / filename
        if p.exists():
            return p
    raise FileNotFoundError(f"Cannot find {filename}. Checked: {[str(d) for d in SEARCH_DIRS]}")

def read_csv(filename: str, parse_dates=None, **kwargs) -> pd.DataFrame:
    path = find_csv(filename)
    return pd.read_csv(path, parse_dates=parse_dates, low_memory=False, **kwargs)

def save_table(df: pd.DataFrame, filename: str, index: bool = False) -> Path:
    path = TABLES_DIR / filename
    df.to_csv(path, index=index)
    print(f"Saved table: {path}")
    return path

def save_fig(name: str, dpi: int = 160):
    path = FIGURES_DIR / name
    plt.tight_layout()
    plt.savefig(path, dpi=dpi, bbox_inches="tight")
    print(f"Saved figure: {path}")
    return path

def money_fmt(x):
    if pd.isna(x):
        return "NA"
    if abs(x) >= 1e9:
        return f"{x/1e9:.2f}B"
    if abs(x) >= 1e6:
        return f"{x/1e6:.2f}M"
    if abs(x) >= 1e3:
        return f"{x/1e3:.2f}K"
    return f"{x:.2f}"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RAW_DIR:", RAW_DIR)
print("INTERIM_DIR:", INTERIM_DIR)
print("MARTS_DIR:", MARTS_DIR)

PROJECT_ROOT: C:\datathon
RAW_DIR: C:\datathon\data\raw
INTERIM_DIR: C:\datathon\data\interim
MARTS_DIR: C:\datathon\data\marts


## 1. Load raw/interim/mart tables

In [2]:
daily = read_csv("5_daily_business_panel.csv", parse_dates=["Date"])
try:
    item_fact = read_csv("1_fact_order_item_enriched.csv", parse_dates=["order_date", "delivery_date", "ship_date", "first_return_date", "first_review_date"])
except Exception:
    items = read_csv("order_items.csv")
    orders = read_csv("orders.csv", parse_dates=["order_date"])
    products = read_csv("products.csv")
    item_fact = items.merge(orders, on="order_id", how="left").merge(products, on="product_id", how="left")
    item_fact["line_revenue"] = item_fact["quantity"] * item_fact["unit_price"]
    item_fact["line_cogs"] = item_fact["quantity"] * item_fact["cogs"]
    item_fact["gross_profit"] = item_fact["line_revenue"] - item_fact["line_cogs"]
    item_fact["gross_margin"] = np.where(item_fact["line_revenue"] > 0, item_fact["gross_profit"] / item_fact["line_revenue"], np.nan)
    item_fact["has_promo"] = item_fact["promo_id"].notna() | item_fact["promo_id_2"].notna()

try:
    order_fact = read_csv("2_fact_order_enriched.csv", parse_dates=["order_date", "ship_date", "delivery_date", "signup_date"])
except Exception:
    orders = read_csv("orders.csv", parse_dates=["order_date"])
    payments = read_csv("payments.csv")
    shipments = read_csv("shipments.csv", parse_dates=["ship_date", "delivery_date"])
    order_fact = orders.merge(payments, on="order_id", how="left", suffixes=("", "_payment")).merge(shipments, on="order_id", how="left")
    order_fact["delivery_lead_days"] = (order_fact["delivery_date"] - order_fact["order_date"]).dt.days

traffic = read_csv("web_traffic.csv", parse_dates=["date"])
customers = read_csv("customers.csv", parse_dates=["signup_date"])
reviews = read_csv("reviews.csv", parse_dates=["review_date"])
returns = read_csv("returns.csv", parse_dates=["return_date"])
products = read_csv("products.csv")
geography = read_csv("geography.csv")
payments = read_csv("payments.csv")
orders = read_csv("orders.csv", parse_dates=["order_date"])

ValueError: Missing column provided to 'parse_dates': 'Date'

## 2. Evidence A — Traffic source quality

Dùng chart này để giải thích vì sao marketing/source mix có thể được dùng làm feature phụ cho forecast hoặc làm giả thuyết cho revenue spikes.

In [ ]:
traffic_source = traffic.groupby("traffic_source", as_index=False).agg(
    sessions=("sessions", "sum"),
    unique_visitors=("unique_visitors", "sum"),
    page_views=("page_views", "sum"),
    avg_bounce_rate=("bounce_rate", "mean"),
    avg_session_duration_sec=("avg_session_duration_sec", "mean")
).sort_values("sessions", ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
ax.scatter(traffic_source["avg_bounce_rate"], traffic_source["avg_session_duration_sec"],
           s=np.sqrt(traffic_source["sessions"]) * 2, alpha=0.65)
for _, r in traffic_source.iterrows():
    ax.text(r["avg_bounce_rate"], r["avg_session_duration_sec"], f" {r['traffic_source']}", va="center")
ax.set_title("Traffic source quality: bounce rate vs session duration")
ax.set_xlabel("Average bounce rate")
ax.set_ylabel("Average session duration (sec)")
save_fig("04_01_traffic_source_quality.png")
plt.show()
traffic_source

## 3. Evidence B — Geo / region contribution

In [ ]:
if "region" in item_fact.columns:
    geo_rev = item_fact.groupby("region", as_index=False).agg(revenue=("line_revenue", "sum"), orders=("order_id", "nunique"), units=("quantity", "sum"))
else:
    item_geo = item_fact.merge(geography[["zip", "region", "district"]], on="zip", how="left")
    geo_rev = item_geo.groupby("region", as_index=False).agg(revenue=("line_revenue", "sum"), orders=("order_id", "nunique"), units=("quantity", "sum"))
geo_rev["revenue_share"] = geo_rev["revenue"] / geo_rev["revenue"].sum()
geo_rev = geo_rev.sort_values("revenue", ascending=True)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.barh(geo_rev["region"], geo_rev["revenue"])
for i, r in geo_rev.reset_index(drop=True).iterrows():
    ax.text(r["revenue"], i, f" {r['revenue_share']:.1%}", va="center")
ax.set_title("Regional revenue contribution")
ax.set_xlabel("Revenue")
ax.set_ylabel("Region")
save_fig("04_02_region_revenue.png")
plt.show()
geo_rev.sort_values("revenue", ascending=False)

## 4. Evidence C — Customer repeat behavior

Phân tích này hữu ích nếu report muốn nhắc đến customer quality hoặc retention, nhưng không nên là trọng tâm nếu đã chọn story revenue–product–inventory.

In [ ]:
customer_orders = (orders.sort_values(["customer_id", "order_date"])
    .assign(prev_order_date=lambda d: d.groupby("customer_id")["order_date"].shift(1))
)
customer_orders["inter_order_gap_days"] = (customer_orders["order_date"] - customer_orders["prev_order_date"]).dt.days
repeat_gap = customer_orders[customer_orders["inter_order_gap_days"].notna()]["inter_order_gap_days"]

fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(repeat_gap.clip(upper=730), bins=40, alpha=0.75)
ax.set_title("Repeat purchase behavior: inter-order gap distribution")
ax.set_xlabel("Days between consecutive purchases, clipped at 730")
ax.set_ylabel("Number of repeat purchase gaps")
save_fig("04_03_inter_order_gap_distribution.png")
plt.show()

repeat_summary = pd.DataFrame({
    "metric": ["repeat_customers", "median_inter_order_gap", "mean_inter_order_gap", "p75_inter_order_gap"],
    "value": [customer_orders["customer_id"].duplicated().sum(), repeat_gap.median(), repeat_gap.mean(), repeat_gap.quantile(0.75)]
})
repeat_summary

## 5. Evidence D — Acquisition channel and age-group order intensity

In [ ]:
orders_by_customer = orders.groupby("customer_id", as_index=False).agg(orders=("order_id", "nunique"))
cust_behavior = customers.merge(orders_by_customer, on="customer_id", how="left")
cust_behavior["orders"] = cust_behavior["orders"].fillna(0)

age_orders = cust_behavior[cust_behavior["age_group"].notna()].groupby("age_group", as_index=False).agg(
    customers=("customer_id", "nunique"),
    total_orders=("orders", "sum"),
    avg_orders_per_customer=("orders", "mean")
).sort_values("avg_orders_per_customer", ascending=False)

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(age_orders["age_group"], age_orders["avg_orders_per_customer"])
ax.set_title("Customer intensity by age group")
ax.set_xlabel("Age group")
ax.set_ylabel("Average orders per customer")
save_fig("04_04_age_group_order_intensity.png")
plt.show()
age_orders

## 6. Evidence E — Payment method and cancellation diagnostic

In [ ]:
pay_cancel = orders.groupby(["payment_method", "order_status"], as_index=False).agg(orders=("order_id", "nunique"))
pay_total = pay_cancel.groupby("payment_method", as_index=False)["orders"].sum().rename(columns={"orders": "total_orders"})
pay_cancel = pay_cancel.merge(pay_total, on="payment_method", how="left")
pay_cancel["status_share"] = pay_cancel["orders"] / pay_cancel["total_orders"]

cancel_view = pay_cancel[pay_cancel["order_status"].eq("cancelled")].sort_values("status_share", ascending=True)

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(cancel_view["payment_method"], cancel_view["status_share"])
ax.set_title("Cancellation rate by payment method")
ax.set_xlabel("Cancelled orders / total orders")
ax.set_ylabel("Payment method")
save_fig("04_05_payment_cancellation_rate.png")
plt.show()
cancel_view

## 7. Evidence F — Fulfillment lead time

In [ ]:
if "delivery_lead_days" not in order_fact.columns:
    order_fact["delivery_lead_days"] = (order_fact["delivery_date"] - order_fact["order_date"]).dt.days
lead = order_fact[order_fact["delivery_lead_days"].notna()].copy()
lead = lead[(lead["delivery_lead_days"] >= 0) & (lead["delivery_lead_days"] <= lead["delivery_lead_days"].quantile(0.995))]

fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(lead["delivery_lead_days"], bins=30, alpha=0.75)
ax.axvline(lead["delivery_lead_days"].median(), linestyle="--", linewidth=2, label=f"Median = {lead['delivery_lead_days'].median():.1f} days")
ax.set_title("Fulfillment evidence: delivery lead-time distribution")
ax.set_xlabel("Days from order to delivery")
ax.set_ylabel("Orders")
ax.legend()
save_fig("04_06_delivery_lead_time_distribution.png")
plt.show()
lead[["delivery_lead_days"]].describe()

## 8. Evidence G — Review rating and return relationship

In [ ]:
review_prod = reviews.merge(products[["product_id", "category", "segment", "size"]], on="product_id", how="left")
return_keys = returns[["order_id", "product_id"]].drop_duplicates().assign(is_returned=1)
review_prod = review_prod.merge(return_keys, on=["order_id", "product_id"], how="left")
review_prod["is_returned"] = review_prod["is_returned"].fillna(0)

rating_return = review_prod.groupby("rating", as_index=False).agg(
    reviews=("review_id", "nunique"),
    returned_review_items=("is_returned", "sum")
)
rating_return["return_rate_among_reviewed"] = rating_return["returned_review_items"] / rating_return["reviews"]

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(rating_return["rating"], rating_return["return_rate_among_reviewed"])
ax.set_title("Product satisfaction proxy: return rate among reviewed items by rating")
ax.set_xlabel("Rating")
ax.set_ylabel("Return rate among reviewed items")
ax.set_xticks(sorted(rating_return["rating"].unique()))
save_fig("04_07_rating_return_relationship.png")
plt.show()
rating_return

## 9. Evidence H — Daily sales anomaly detection

Các ngày anomaly có thể dùng để kiểm tra campaign, stockout, payment/cancellation hoặc data issue.

In [ ]:
sales_ts = daily[["Date", "Revenue", "COGS"]].copy().sort_values("Date")
sales_ts["revenue_28d_mean"] = sales_ts["Revenue"].rolling(28, min_periods=7).mean()
sales_ts["revenue_28d_std"] = sales_ts["Revenue"].rolling(28, min_periods=7).std()
sales_ts["z_score"] = (sales_ts["Revenue"] - sales_ts["revenue_28d_mean"]) / sales_ts["revenue_28d_std"]
sales_ts["is_anomaly"] = sales_ts["z_score"].abs() >= 3

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(sales_ts["Date"], sales_ts["Revenue"], linewidth=1, alpha=0.65, label="Daily revenue")
anom = sales_ts[sales_ts["is_anomaly"]]
ax.scatter(anom["Date"], anom["Revenue"], s=35, label="|z| >= 3")
ax.set_title("Daily revenue anomaly screen using rolling z-score")
ax.set_xlabel("Date")
ax.set_ylabel("Revenue")
ax.legend()
save_fig("04_08_daily_revenue_anomalies.png")
plt.show()

anomaly_table = anom.sort_values("z_score", key=lambda s: s.abs(), ascending=False).head(30)
save_table(anomaly_table, "04_top_revenue_anomalies.csv")
anomaly_table

## 10. Evidence catalog for GitHub README

In [ ]:
evidence_catalog = pd.DataFrame([
    {"figure": "04_01_traffic_source_quality.png", "use_in_report": "optional", "supports": "Marketing/source quality and forecast features"},
    {"figure": "04_02_region_revenue.png", "use_in_report": "optional", "supports": "Geo contribution"},
    {"figure": "04_03_inter_order_gap_distribution.png", "use_in_report": "appendix", "supports": "Customer repeat behavior"},
    {"figure": "04_04_age_group_order_intensity.png", "use_in_report": "appendix", "supports": "Customer segmentation"},
    {"figure": "04_05_payment_cancellation_rate.png", "use_in_report": "appendix", "supports": "Payment and cancellation diagnostic"},
    {"figure": "04_06_delivery_lead_time_distribution.png", "use_in_report": "optional", "supports": "Fulfillment performance"},
    {"figure": "04_07_rating_return_relationship.png", "use_in_report": "appendix", "supports": "Review/return quality signal"},
    {"figure": "04_08_daily_revenue_anomalies.png", "use_in_report": "optional", "supports": "Anomaly detection before forecasting"},
])
save_table(evidence_catalog, "04_supporting_evidence_catalog.csv")
evidence_catalog

## 11. How to use this notebook

Không nhét tất cả chart vào report. Repo/GitHub nên trình bày:

- `03_eda_main_storyline.ipynb`: phần chính, có insight mạnh và recommendation.
- `04_eda_supporting_evidence.ipynb`: bằng chứng phụ, dùng khi cần giải thích thêm hoặc đưa vào appendix.